# Deadlock 

In [1]:
!pip3 install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 45.9 MB/s  0:00:00 eta 0:00:01


In [17]:
import duckdb
import requests

# 1. Fetch manifest and initialize DuckDB connection
manifest = requests.get("https://data.deadlock-api.com/v1/manifest.json").json()
base = manifest["public_url"]

def urls(table: str, limit: int = 50) -> list[str]:
    t = manifest["tables"][table]
    gen = t.get("generation", 0)
    file_list = [f"{base}/{f['key']}" for f in t["files"]
                 if t["policy"] == "snapshot" or f.get("generation", 0) == gen]
    # Take only the latest N files to prevent OOM errors during schema parsing
    return file_list[-limit:]

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Prevent OOM crashes
con.execute("SET threads = 2;")
con.execute("SET memory_limit = '4GB';")
con.execute("SET preserve_insertion_order = false;")

print("Available tables in manifest:", list(manifest["tables"].keys()))

# 2. Create views using a safe subset of files
target_tables = ["match_salts", "match_player", "steam_profiles"]

for name in target_tables:
    if name in manifest["tables"]:
        file_urls = urls(name, limit=30)  # Capped at 30 files for testing
        con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet({file_urls!r}, union_by_name = true)")

Available tables in manifest: ['hero_leaderboard', 'leaderboard', 'match_player', 'match_salts', 'steam_profile_observed_names', 'steam_profiles']


In [18]:
print(con.sql("DESCRIBE match_salts;").df())

# Check column names in the match_player view
print(con.sql("DESCRIBE steam_profiles;").df())

     column_name               column_type null   key default extra
0       match_id                   UBIGINT  YES  None    None  None
1     cluster_id                  UINTEGER  YES  None    None  None
2  metadata_salt                  UINTEGER  YES  None    None  None
3    replay_salt                  UINTEGER  YES  None    None  None
4     created_at  TIMESTAMP WITH TIME ZONE  YES  None    None  None
             column_name                 column_type null   key default extra
0             account_id                    UINTEGER  YES  None    None  None
1            personaname                     VARCHAR  YES  None    None  None
2             profileurl                     VARCHAR  YES  None    None  None
3                 avatar                     VARCHAR  YES  None    None  None
4           personastate                     VARCHAR  YES  None    None  None
5               realname                     VARCHAR  YES  None    None  None
6            countrycode                     V

In [6]:
match_player_cols = con.sql("DESCRIBE match_player;").df()["column_name"]
print(sorted([col for col in match_player_cols.tolist() if "team" in col], reverse = False))
print(match_player_cols.shape)

['average_badge_team0', 'average_badge_team1', 'mid_boss.team_claimed', 'mid_boss.team_killed', 'objectives.team', 'objectives.team_objective', 'objectives_mask_team0', 'objectives_mask_team1', 'stats.teammate_barriering', 'stats.teammate_healing', 'street_brawl_rounds.winning_team', 'team', 'team0_tracked_stats', 'team1_tracked_stats', 'team_score', 'winning_team']
(157,)


In [21]:
match_player = con.sql("select match_id from match_player;").df()
match_player.head()

,match_id
0,123875483
1,123875483
2,123875483
3,123875483
4,123875483


In [ ]:
match_player

In [22]:
match_player.shape

(7659613, 1)

In [23]:
match_salts_cols = con.sql("DESCRIBE match_salts;").df()["column_name"]
print(sorted(match_salts_cols.tolist(), reverse = False))
print(match_salts_cols.shape)

['cluster_id', 'created_at', 'match_id', 'metadata_salt', 'replay_salt']
(5,)


In [ ]:
# 2. Player-Level Extraction Query
player_query = """
    SELECT 
        m.match_id,
        mp.account_id,
        mp.hero_id,
        mp.team,
        mp.won,
        mp.net_worth,
        mp.stats.net_worth,
        mp.stats.gold_lane_creep,
        mp.stats.gold_neutral_creep,
        mp.stats.gold_player,
        mp.kills,
        mp.deaths,
        mp.assists,
        mp.stats.player_damage,
        mp.stats.player_damage_taken,
        mp.stats.ability_kills,
        mp.stats.headshot_kills,
        mp.stats.melee_kills,
        mp.stats.boss_damage,
        mp.stats.creep_damage,
        mp.stats.creep_kills,
        mp.items.item_id,
        mp.items.game_time_s,
        mp.items.sold_time_s,
        CASE WHEN m.team = mp.winning_team THEN 1 ELSE 0 END AS label
    FROM match_salts m
    JOIN match_player mp ON m.match_id = mp.match_id
    LIMIT 100000;
"""

# 3. Load into Pandas
df_players = con.sql(player_query).df()

df_players.head()

BinderException: Binder Error: Values list "mp" does not have a column named "stats"

In [ ]:
import duckdb
import requests
from pathlib import Path

# 1. Initialize DuckDB connected to a persistent local file for DBeaver
data_dir = Path("../data")
data_dir.mkdir(parents = True, exist_ok = True)
db_path = str(data_dir / "deadlock.duckdb")

con = duckdb.connect(db_path)
con.execute("SET threads = 1;")
con.execute("SET memory_limit = '8GB';")
con.execute("SET preserve_insertion_order = false;")

# 2. Fetch manifest
manifest = requests.get("https://data.deadlock-api.com/v1/manifest.json").json()
base = manifest["public_url"]

def get_file_slice(table: str, start_idx: int, end_idx: int) -> list[str]:
    """Helper to pull a specific non-overlapping slice of parquet files."""
    t = manifest["tables"][table]
    gen = t.get("generation", 0)
    file_list = [f"{base}/{f['key']}" for f in t["files"]
                 if t["policy"] == "snapshot" or f.get("generation", 0) == gen]
    return file_list[start_idx:end_idx]

# Define your batch window (e.g., files 0 to 30 for Batch 1, then 30 to 60 for Batch 2)
START_FILE = 0
END_FILE = 15

target_tables = ["match_salts", "match_player"]

def extract_match_player_salts_data(iterations = 1, start_file_no = START_FILE, end_file_no = END_FILE, target_tables = target_tables):
    start_no = start_file_no
    end_no = end_file_no

    for name in target_tables:
        if name in manifest["tables"]:
            file_urls = get_file_slice(name, start_no, end_no)
            if not file_urls: 
                print(f"No more files available for {name} at index {start_no}. Skipping this table.")
                continue

            print(f"Creating view for {name} using files {start_no} to {end_no}...")
            con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet({file_urls!r}, union_by_name = true)")

            
    # 3. Extract and insert into a persistent master table safely (avoiding duplicates via match_id)
    print("Extracting and inserting player features into persistent table...")
    con.execute("""
        CREATE TABLE IF NOT EXISTS player_features AS 
        SELECT 
            m.match_id,
            mp.account_id,
            mp.hero_id,
            mp.team,
            mp.won,
            mp.net_worth,
            mp."stats.net_worth",
            mp."stats.gold_lane_creep",
            mp."stats.gold_neutral_creep",
            mp."stats.gold_player",
            mp.kills,
            mp.deaths,
            mp.assists,
            mp."stats.player_damage",
            mp."stats.player_damage_taken",
            mp."stats.ability_kills",
            mp."stats.headshot_kills",
            mp."stats.melee_kills",
            mp."stats.boss_damage",
            mp."stats.creep_damage",
            mp."stats.creep_kills",
            mp."items.item_id",
            mp."items.game_time_s",
            mp."items.sold_time_s",
            CASE WHEN mp.team = mp.winning_team THEN 1 ELSE 0 END AS label
        FROM match_salts m
        JOIN match_player mp ON m.match_id = mp.match_id
    """)

    # If the master table already exists from a previous run, use an INSERT with a duplicate check:
    con.execute("""
        INSERT INTO player_features
        SELECT 
            m.match_id,
            mp.account_id,
            mp.hero_id,
            mp.team,
            mp.won,
            mp.net_worth,
            mp."stats.net_worth",
            mp."stats.gold_lane_creep",
            mp."stats.gold_neutral_creep",
            mp."stats.gold_player",
            mp.kills,
            mp.deaths,
            mp.assists,
            mp."stats.player_damage",
            mp."stats.player_damage_taken",
            mp."stats.ability_kills",
            mp."stats.headshot_kills",
            mp."stats.melee_kills",
            mp."stats.boss_damage",
            mp."stats.creep_damage",
            mp."stats.creep_kills",
            mp."items.item_id",
            mp."items.game_time_s",
            mp."items.sold_time_s",
            CASE WHEN mp.team = mp.winning_team THEN 1 ELSE 0 END AS label
        FROM match_salts m
        JOIN match_player mp ON m.match_id = mp.match_id
        WHERE m.match_id NOT IN (SELECT DISTINCT match_id FROM player_features);
    """)
    
    print("Running inspection queries...")
    stats_query = """
        SELECT
            COUNT(DISTINCT match_id) AS total_unique_matches,
            COUNT(*) AS total_player_rows,
            COUNT(*) / NULLIF(COUNT(DISTINCT match_id), 0) AS avg_players_per_match
        FROM player_features
    """
    print(con.sql(stats_query).df())

    print("Keeping only matches with 12 players...")
    full_match_query = """
        CREATE OR REPLACE TABLE player_features AS
        SELECT pf.*
        FROM player_features pf
        JOIN (
            SELECT match_id
            FROM player_features
            GROUP BY match_id
            HAVING COUNT(*) = 12
        ) complete_matches ON pf.match_id = complete_matches.match_id
    """
    con.execute(full_match_query)
    print("Partial match cleanup complete.")

    print("Re-running inspection queries...")
    print(con.sql(stats_query).df())
    con.close()
    return

In [4]:
extract_match_player_salts_data(1, 0, 15)

Creating view for match_salts using files 0 to 15...
Creating view for match_player using files 0 to 15...
Extracting and inserting player features into persistent table...
Running inspection queries...
   total_unique_matches  total_player_rows  avg_players_per_match
0                 29178             341387              11.700151
Keeping only matches with 12 players...
Partial match cleanup complete.
Re-running inspection queries...
   total_unique_matches  total_player_rows  avg_players_per_match
0                 27947             335364                   12.0
